# Estimator statistics

In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from master.fft import *
import jax
from master.symmetry import *
from master.phase_retrieval import *
from master.obj import *
from master.plotting import plot_intensity_phase, plot
import master.device
from master.ll_plotting import *

jax.config.update("jax_enable_x64", True)
cdtype = jnp.array(1j).dtype
gpu_idx, free_mb, gpu_count = master.device.setup_device(device_id=1)

In [ ]:
# N_crop = 127
# N_pad = 128
N_crop = 256
N_pad = 128
# N_crop = 33
# N_pad = 33
N_samples = 32
N_photons = 2.5e11
# x_obj, obj_support = crop_and_pad(as_phase_field(cameraman()), N_crop=N_crop, N_pad=N_pad)
x_obj, obj_support = crop_and_pad(cameraman(), N_crop=N_crop, N_pad=N_pad)
x, y, X, Y, R, k_0, k_x, k_y, K_X, K_Y, k, r_extent, k_extent = create_physical_axes(
    x_obj
)

use_cos_filter = False
filter_I = True

if use_cos_filter:
    i = np.arange(0, N_crop, dtype=np.float64)
    j = np.arange(0, N_crop, dtype=np.float64)
    I, J = np.meshgrid(i, j)
    filt = np.cos(jnp.pi / 2 * (I / (N_crop / 2) - 1)) * np.cos(
        jnp.pi / 2 * (J / (N_crop / 2) - 1)
    )
    if filter_I:
        filt = jnp.sqrt(filt)

    # x_obj = jnp.ones_like(x_obj)
    x_obj, _ = crop_and_pad(
        center_crop(x_obj, N_crop) * filt,
        N_crop=N_crop,
        N_pad=N_pad,
    )
x_obj = normalize_image_intensity(x_obj)
plot_intensity_phase(x_obj, r_extent, label="y", vertical=True)

In [ ]:
F_pupil = jnp.ones_like(k)

spectral_oversampling = analyse_oversampling(F_pupil)
obj_oversampling = analyse_oversampling(obj_support)
print(
    f"Spectral oversampling: {spectral_oversampling};  Object oversampling: {obj_oversampling}"
)
y_obj = ifft2_shifted(x_obj)
x_obj_ph = np.sqrt(N_photons) * x_obj
plot_intensity_phase(y_obj, k_extent, label=r"\hat{x'}", use_log=True)

In [ ]:
y = fft2_shifted(y_obj)
plot_intensity_phase(y, r_extent, label="y")

In [ ]:
from numpy.testing import assert_array_almost_equal

# Verify that y and x_obj are the same in this setting, implicitly checks that x=fft2_shifted(ifft2_shifted(x)) so that
# FFT operators are unitary as they should be
assert_array_almost_equal(y, x_obj)

In [ ]:
from master.noise import *

yhat_mu, y_d = sample_field(y_obj, N_photons=N_photons, batch_dim=N_samples)
# y_mu, _ = sample_field(y, N_photons=N_photons, batch_dim=1)
y_mu = y * jnp.sqrt(N_photons)
jnp.max(y_d), jnp.median(y_d), jnp.mean(y_d), jnp.min(y_d)

In [ ]:
for i in range(min(N_samples, 5)):
    plot(y_d[i, :], use_log=True)

In [ ]:
# Fix base_key to keep estimator itself deterministic given yhat_sample. Avoids difficulties with interpretation
# of statistics across samples
base_key = jax.random.PRNGKey(1776)
# y_est_raw, converged, err_L2 = difference_map(
#     jnp.abs(yhat_sample),
#     obj_support,
#     restarts=1,
#     epsilon=1e-7,
#     iterations=2000,
#     stagnate_rel_delta=1e-12,
#     beta=0.975,
#     base_key=base_key,
# )
x_est_raw, converged, err_L2 = hio(
    jnp.abs(y_d),
    obj_support,
    iterations=250,
    beta=0.9,
    base_key=base_key,
)
x_est_raw, converged, err_L2 = error_reduction(
    jnp.abs(y_d),
    obj_support,
    iterations=250,
    initial_field=x_est_raw,
)
x_est_hio = x_est_raw

## Select representative

In [ ]:
rescale_energy = False
x_est = np.zeros_like(x_est_raw)
x_est = choose_representative_broadcast(x_est_raw * obj_support, x_obj_ph)

## Estimator statistics

In [ ]:
import numpy as np

final_complex = x_est
final = np.ones_like(x_est)
final_spectrum = np.ones_like(x_est)
error_L2 = np.ones(N_samples)
for i in range(N_samples):
    y_abs_est = jnp.abs(fft2_shifted(x_est[i, :]))
    error_L2[i] = jnp.sum(jnp.abs(y_abs_est - y_d[i, :]) ** 2) / jnp.sum(
        jnp.abs(y_d[i, :]) ** 2
    )
    print(error_L2[i])
    final[i, :] = final_complex[i, :]
    final_spectrum[i, :] = ifft2_shifted(final_complex[i, :])

In [ ]:
import matplotlib.pyplot as plt

plt.plot(error_L2, ".")
plt.ylabel(r"$\epsilon_{L2}=\frac{|\hat{y} - y|^2}{|y|^2}$")

In [ ]:
from sklearn.cluster import KMeans


def determine_converged(error_L2, N_samples, discrimination_factor=1.2):
    kmeans = KMeans(n_clusters=2).fit(error_L2.reshape(-1, 1))
    labels = kmeans.labels_
    centers = kmeans.cluster_centers_
    max_ix = np.argmax(centers)
    min_ix = np.argmin(centers)

    #  If the two found clusters are quite close in error, merge
    if centers[max_ix] > discrimination_factor * centers[min_ix]:
        if max_ix == 1:
            indices = (1.0 - labels).nonzero()[0]
        else:
            indices = labels.nonzero()[0]
    else:
        indices = np.arange(0, N_samples)
    conv_count = indices.size
    conv_rate = conv_count / N_samples
    return centers, indices, conv_count, conv_rate


centers, indices, conv_count, conv_rate = determine_converged(error_L2, N_samples)
print(
    f"Found centers are {centers.flatten()}\n{conv_count}/{N_samples} converged ({conv_rate*100:.3g}%) "
)

In [ ]:
print(f"Estimator statistics over {conv_count}/{N_samples} converged samples")
final_conv = final[indices]
n_hat = np.abs(final_conv) ** 2
n_hat_mu = np.mean(n_hat, axis=0)
n_hat_var = np.var(n_hat, axis=0, ddof=1)
n_hat_std = np.std(n_hat, axis=0, ddof=1)

plot(n_hat_mu, label=r"$[\mu_{\hat{n}_k}]^{HIO}$")
plot(jnp.abs(y_mu) ** 2, label=r"$n_k$")
plot(n_hat_mu - jnp.abs(y_mu) ** 2, label=r"$[\mu_{\hat{n}_k}]^{HIO} - n_k$")
plot(n_hat_std, label=r"$[\sigma_{\hat{n}_k}]^{HIO}$")

In [ ]:
# TODO: explicitly consider wrapping? Averaging over +-pi might mess up things, need to verify whether this type of mixing occurs
phi_hat = np.angle(final_conv) * obj_support
phi_hat_mu = np.mean(phi_hat, axis=0)
phi_hat_var = np.var(phi_hat, axis=0, ddof=1)
phi_hat_std = np.std(phi_hat, axis=0, ddof=1)

plot(phi_hat_mu, label=r"$[\mu_{\hat{\phi}_k}]^{HIO}$")
plot(jnp.angle(y_mu) * obj_support, label=r"$\phi_k$")
plot(
    phi_hat_mu - jnp.angle(y_mu) * obj_support,
    label=r"$[\mu_{\hat{\phi}_k}]^{HIO} - \phi_k$",
)
plot(phi_hat_std, label=r"$[\sigma_{\hat{\phi}_k}]^{HIO}$")

## Maximum likelihood refinement

In [ ]:
from master.mle import *

y_d_indices = jnp.abs(y_d[indices])
x_model = final_conv
y_model = ifft2_shifted(final_conv)
ll = log_likelihood(y_d_indices, y_model)
ll

# First order ML refinement

In [ ]:
alpha = 2e-2
epsilon = 1e-12
N_iter = 1000
conv_count = final_conv.shape[0]
lls_first_order = np.zeros((N_iter, conv_count))
x_model_ml_first_order = final_conv

x_model_ml_first_order, lls_first_order = ml_refinement_first_order(
    final_conv,
    y_d_indices,
    obj_support,
    alpha=alpha,
    epsilon=epsilon,
    N_iter=N_iter,
    track_ll=True,
)

In [ ]:
norm = construct_norm_ll(lls_first_order)
plot_ll(lls_first_order, f_norm=norm)

In [ ]:
def estimator_stats(x_hat, x_true):
    mu = np.mean(x_hat, axis=0)
    std = np.std(x_hat, axis=0, ddof=1)
    bias = mu - x_true
    mse = std**2 + bias**2
    return mu, std, bias, mse


def evaluate_estimator(x_model_ml):
    print(f"Estimator statistics over {conv_count}/{N_samples} converged samples")
    n_hat = np.abs(x_model_ml) ** 2
    n = jnp.abs(y_mu) ** 2
    n_hat_mu, n_hat_std, n_hat_bias, n_hat_MSE = estimator_stats(n_hat, n)
    print("n_k")
    plot(n_hat_mu, label=r"$[\mu_{\hat{n}_k}]^{HIO+ML}$")
    plot(n, label=r"$n_k$")
    plot(n_hat_bias, label=r"$[\mu_{\hat{n}_k}]^{HIO+ML} - n_k$")
    plot(n_hat_std, label=r"$[\sigma_{\hat{n}_k}]^{HIO+ML}$")
    plot(n_hat_MSE, label=r"$[MSE_{\hat{n}_k}]^{HIO+ML}$")
    # TODO: explicitly consider wrapping? Averaging over +-pi might mess up things, need to verify whether this type of mixing occurs
    print("phi_k")

    phi_hat = np.angle(x_model_ml) * obj_support
    phi = jnp.angle(y_mu) * obj_support
    phi_hat_mu, phi_hat_std, phi_hat_bias, phi_hat_MSE = estimator_stats(phi_hat, phi)
    plot(phi_hat_mu, label=r"$[\mu_{\hat{\phi}_k}]^{HIO+ML}$")
    plot(phi, label=r"$\phi_k$")
    plot(
        phi_hat_bias,
        label=r"$[\mu_{\hat{\phi}_k}]^{HIO+ML} - \phi_k$",
    )
    plot(phi_hat_std, label=r"$[\sigma_{\hat{\phi}_k}]^{HIO+ML}$")
    plot(phi_hat_MSE, label=r"$[MSE_{\hat{\phi}_k}]^{HIO+ML}$")


evaluate_estimator(x_model_ml_first_order)

In [ ]:
import h5py


def save_est_to_h5(filename, x_hat):
    with h5py.File(filename, "w") as f:
        x_hat = x_hat
        x_true = y_mu * obj_support
        f.create_dataset("x_est", data=x_hat)
        f.create_dataset("x_true", data=x_true)
        f.attrs["N_crop"] = N_crop
        f.attrs["N_pad"] = N_pad
        f.attrs["N_photons"] = N_photons
        f.attrs["N_samples"] = N_samples


save_est_to_h5(
    f"data/estimates-hio-first-order-ml-{N_photons:.3g}.h5",
    x_model_ml_first_order * obj_support,
)

## Second order ML refinement: Fisher scoring in two ways

In [ ]:
x_model_ml_second_order, lls_second_order = ml_fisher_scoring_unitary_A(
    final_conv,
    y_d_indices,
    obj_support,
    alpha=1.0,
    epsilon=epsilon,
    N_iter=N_iter,
    track_ll=True,
)

In [ ]:
plot_ll(lls_second_order, f_norm=norm)

In [ ]:
compare_lls(lls_second_order, lls_first_order)

In [ ]:
compare_lls(
    lls_second_order,
    lls_first_order,
    geometric_mean=True,
    labels=["Second order", "First order"],
)

In [ ]:
evaluate_estimator(x_model_ml_second_order)

# Newton-Raphson with conjugate gradient

In [ ]:
x_model_ml_second_order_alt, lls_second_order_alt = ml_fisher_scoring_general_A(
    final_conv,
    yhat_sample_conv,
    obj_support,
    alpha=0.5,
    epsilon=epsilon,
    N_iter=1000,
    track_ll=True,
)

In [ ]:
plot_ll(lls_second_order_alt)

In [ ]:
save_est_to_h5(
    "data/estimates-hio-second-order-ml.h5", x_model_ml_second_order_alt * obj_support
)

In [ ]:
compare_lls(
    lls_second_order,
    lls_second_order_alt,
    geometric_mean=True,
    labels=["FS using analytical form", "FS using matrix solve"],
)

In [ ]:
evaluate_estimator(x_model_ml_second_order_alt)